### Loading libraries


In [1]:
import pandas as pd
import numpy as np 

### Reading CSV files

In [2]:
# --- 0. Define the file path ---
# This is the folder where all your files are located.
base_path = '/Users/hrichaacharya/Desktop/hope/'

# --- 1. Load all the necessary CSV files ---
# Load the inspection, hive_info, apiary_info and weather data files
inspections_df = pd.read_csv(base_path + 'HCC_Inspections.csv')
hive_info_df = pd.read_csv(base_path + 'Hive_Information.csv')
apiary_info_df = pd.read_csv(base_path + 'Apiary_Information.csv')
weather_NC_df = pd.read_csv(base_path + 'weather_NC.csv')
weather_UT_df = pd.read_csv(base_path + 'weather_UT.csv')

### Combining and cleaning datasets

This code loads three separate honeybee datasets (apiary, hive, and inspection details) and merges them into one large, combined table. It then cleans this master table by converting dates, dropping some unneeded columns, and removing any rows that have missing values or are duplicates, resulting in a clean dataset ready for analysis.

In [3]:
apiary_df = pd.read_csv(base_path + 'Apiary_Information.csv')
hive_df = pd.read_csv(base_path + 'Hive_Information.csv')
inspection_df = pd.read_csv(base_path + 'HCC_Inspections.csv')
hive_merged = hive_df.merge(apiary_df, on='ApiaryID', how='left')
inspect_merged = inspection_df.merge(hive_merged, on='HiveID', how='left')
inspect_merged['InsptDate'] = pd.to_datetime(inspect_merged['InsptDate'])
inspections_with_state = inspect_merged.copy(deep=True)
inspections_with_state = inspections_with_state.drop(columns=['InpsectionID','Percent_Met','Hive_Tag','Apiary'])
inspections_with_state.dropna(inplace=True)
inspections_with_state.drop_duplicates(inplace=True)
inspections_with_state.reset_index(drop=True, inplace=True)
inspections_with_state = inspections_with_state.dropna(subset=['InsptDate'])
inspections_with_state.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2087 entries, 0 to 2086
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   HiveID     2087 non-null   int64         
 1   InsptDate  2087 non-null   datetime64[ns]
 2   Brood      2087 non-null   float64       
 3   Bees       2087 non-null   float64       
 4   Queen      2087 non-null   float64       
 5   Food       2087 non-null   float64       
 6   Stressors  2087 non-null   float64       
 7   Space      2087 non-null   float64       
 8   Healthy    2087 non-null   object        
 9   ApiaryID   2087 non-null   int64         
 10  City       2087 non-null   object        
 11  State      2087 non-null   object        
dtypes: datetime64[ns](1), float64(6), int64(2), object(3)
memory usage: 195.8+ KB


In [4]:
print(inspections_with_state.head())

   HiveID  InsptDate  Brood  Bees  Queen  Food  Stressors  Space Healthy  \
0       1 2016-06-15    1.0   1.0    1.0   1.0        0.0    1.0      No   
1       1 2016-07-22    1.0   1.0    1.0   1.0        0.0    1.0      No   
2       1 2016-08-01    1.0   1.0    1.0   1.0        1.0    1.0     Yes   
3       1 2016-08-08    1.0   1.0    1.0   1.0        1.0    1.0     Yes   
4       1 2016-08-15    1.0   1.0    1.0   1.0        0.0    1.0      No   

   ApiaryID    City State  
0         1  Durham    NC  
1         1  Durham    NC  
2         1  Durham    NC  
3         1  Durham    NC  
4         1  Durham    NC  


In [5]:
weather_NC_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1826 entries, 0 to 1825
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   STATION  1826 non-null   object 
 1   NAME     1826 non-null   object 
 2   DATE     1826 non-null   object 
 3   AWND     1826 non-null   float64
 4   PRCP     1826 non-null   float64
 5   SNOW     1826 non-null   float64
 6   TAVG     1826 non-null   int64  
 7   TMAX     1826 non-null   int64  
 8   TMIN     1826 non-null   int64  
 9   TSUN     7 non-null      float64
dtypes: float64(4), int64(3), object(3)
memory usage: 142.8+ KB


In [6]:
weather_UT_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1826 entries, 0 to 1825
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   STATION  1826 non-null   object 
 1   NAME     1826 non-null   object 
 2   DATE     1826 non-null   object 
 3   AWND     1826 non-null   float64
 4   PRCP     1826 non-null   float64
 5   SNOW     1825 non-null   float64
 6   TAVG     1826 non-null   int64  
 7   TMAX     1826 non-null   int64  
 8   TMIN     1826 non-null   int64  
 9   TSUN     2 non-null      float64
dtypes: float64(4), int64(3), object(3)
memory usage: 142.8+ KB


### Feature Engineering

This code performs feature engineering by linking weather data to bee inspections. First, it prepares all datasets by converting date columns to a proper datetime format, setting the weather dates as an index for faster lookups. Its main action is to apply a custom function to every inspection row; this function calculates 7-day trailing weather features (like average temperature, wind, and precipitation) for the week before the inspection date, based on the hive's state. Finally, it adds these new calculated weather features as new columns to the main inspection DataFrame and saves the combined dataset to a new CSV file.

In [7]:
# --- 1. Prepare DataFrames for Time-Series Analysis ---
# This is a critical step. We must convert date strings into actual datetime objects to perform date calculations.

# Convert inspection dates
inspections_with_state['InsptDate'] = pd.to_datetime(inspections_with_state['InsptDate'], errors='coerce')

# Convert weather dates
weather_NC_df['DATE'] = pd.to_datetime(weather_NC_df['DATE'])
weather_UT_df['DATE'] = pd.to_datetime(weather_UT_df['DATE'])

# Set the 'DATE' column as the index for weather data.
# This makes date-based lookups (slicing) much, much faster.
weather_NC_df = weather_NC_df.set_index('DATE')
weather_UT_df = weather_UT_df.set_index('DATE')

In [8]:
# --- Define the Feature Engineering Function ---
def get_weather_features(row):
    """
    Calculates 7-day trailing weather features for a single inspection row.
    """
    # Create a Series of NaNs (blanks) to return by default
    feature_names = ['Avg_prcp', 'Avg_wind', 'Avg_tmax', 'Avg_tmin', 'Avg_tavg', 'Avg_snow', 'Num_frost_days']
    nan_series = pd.Series([np.nan] * 7, index=feature_names)

    # --- A. Select the correct weather dataframe ---
    if row['State'] == 'NC':
        weather_data = weather_NC_df
    elif row['State'] == 'UT':
        weather_data = weather_UT_df
    else:
        # If State is not NC (or UT), return the blanks
        return nan_series

    # --- B. Define the 7-day window ---
    # We want the 7 days *before* the inspection date.
    # E.g., if inspect is on Jan 8, we want Jan 1 - Jan 7.
    inspection_date = row['InsptDate']
    end_date = inspection_date - pd.Timedelta(days=1)
    start_date = inspection_date - pd.Timedelta(days=7)

    # --- C. Slice the weather data for that window ---
    # We use .loc[] to slice the indexed weather data
    window_data = weather_data.loc[start_date:end_date]

    # If the window has no data (e.g., missing dates), return blanks
    if window_data.empty:
        return nan_series

    # --- D. Calculate the new features ---
    avg_prcp = window_data['PRCP'].mean()
    avg_wind = window_data['AWND'].mean()
    avg_tmax = window_data['TMAX'].mean()
    avg_tmin = window_data['TMIN'].mean()
    avg_tavg = window_data['TAVG'].mean()
    avg_snow = window_data['SNOW'].mean()
    
    # For Num_frost_days, we count the rows where TAVG was < 0
    num_frost_days = window_data[window_data['TMIN'] < 32].shape[0]

    # --- E. Return the new features as a Series ---
    return pd.Series([
        avg_prcp, avg_wind, avg_tmax, avg_tmin, avg_tavg, avg_snow, num_frost_days
    ], index=feature_names)

In [9]:
# --- Apply the Function to Every Row ---
# .apply(..., axis=1) runs our function once for each row
inspections_with_state = inspections_with_state.drop(columns=['Brood','Bees','Queen','Food','Stressors','Space'])
new_weather_features = inspections_with_state.apply(get_weather_features, axis=1)

# --- Join the New Features Back to the Main DataFrame ---
# pd.concat joins the two dataframes side-by-side
inspections_with_state = pd.concat([inspections_with_state, new_weather_features], axis=1)

print("\nSuccessfully added 7 new weather features.")
print("Here is a preview of the final data (scroll right to see new columns):")
inspections_with_state.info()
print(inspections_with_state.head(5))
print(inspections_with_state.tail(5))


Successfully added 7 new weather features.
Here is a preview of the final data (scroll right to see new columns):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2087 entries, 0 to 2086
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   HiveID          2087 non-null   int64         
 1   InsptDate       2087 non-null   datetime64[ns]
 2   Healthy         2087 non-null   object        
 3   ApiaryID        2087 non-null   int64         
 4   City            2087 non-null   object        
 5   State           2087 non-null   object        
 6   Avg_prcp        2087 non-null   float64       
 7   Avg_wind        2087 non-null   float64       
 8   Avg_tmax        2087 non-null   float64       
 9   Avg_tmin        2087 non-null   float64       
 10  Avg_tavg        2087 non-null   float64       
 11  Avg_snow        2087 non-null   float64       
 12  Num_frost_days  2087 non-null   float64      

In [10]:
output_file_path = f"{base_path}final_weather_features.csv"
inspections_with_state.to_csv(output_file_path, index=False)